# 2.0 Clean and Join

**Class 2:** turn the raw files from notebook 1.1 (three tables, each saved separately for the study and control groups) into **one analysis-ready table** with one row per person.

| Step | What we do | Main functions |
|---|---|---|
| 1 | Load the six raw files | `load_data()` |
| 2 | Create a **flagging variable**: which group is each person in? | `mutate()` |
| 3 | **Bring the groups together** (stack them) | `bind_rows()` |
| 4 | Survey: tidy answers, one column per question | `mutate()`, `pivot_wider()` |
| 5 | Measurements: group names, keep the latest value, one column each | `case_when()`, `slice_max()`, `pivot_wider()` |
| 6 | Join everything and add age and age group | `left_join()`, `cut()` |
| 7 | Save `final_analysis.csv` | `save_data()` |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

## Step 1: Load the raw data

Notebook 1.1 saved each group to its own file: `_study` (has diabetes) and `_control` (no diabetes).

In [ ]:
person_study        <- load_data("person_study.csv")
person_control      <- load_data("person_control.csv")
survey_study        <- load_data("survey_study.csv")
survey_control      <- load_data("survey_control.csv")
measurement_study   <- load_data("measurement_study.csv")
measurement_control <- load_data("measurement_control.csv")

glimpse(person_study)

## Step 2: Create a flagging variable

Once the groups are combined, we need to know **which group each person came from**. A **flagging variable** records it: here a new column, `diabetes`, set to `"Yes"` for everyone in the study group and `"No"` for everyone in the control group.

`mutate()` adds a column. Giving it a single value fills that value into every row.

In [ ]:
person_study   <- person_study   %>% mutate(diabetes = "Yes")
person_control <- person_control %>% mutate(diabetes = "No")

head(person_study, 3)
head(person_control, 3)

We only flag the **person** tables. The survey and measurement tables are linked to a group through `person_id` when we join everything in Step 6.

## Step 3: Bring the groups together

The study and control tables have the **same columns**, so we **stack** them: `bind_rows()` puts the control rows underneath the study rows.

| | `bind_rows()` (stacking) | `left_join()` (joining) |
|---|---|---|
| Adds | **rows**: more people, same columns | **columns**: more variables for the same people |
| Use when | the tables have the same columns | the tables share an ID (`person_id`) |
| Here | study + control groups | demographics + survey + measurements (Step 6) |

In [ ]:
person <- bind_rows(person_study, person_control) %>%
  transmute(person_id, diabetes, date_of_birth = as.Date(date_of_birth), gender, race, ethnicity)

survey      <- bind_rows(survey_study, survey_control)
measurement <- bind_rows(measurement_study, measurement_control)

person %>% count(diabetes)

**Check:** nobody should be in **both** groups. `intersect()` finds person IDs that appear in both tables; the answer should be **0**.

In [ ]:
length(intersect(person_study$person_id, person_control$person_id))

## Step 4: Survey answers

Right now `survey` is **long**: one row per person *per question*. We want **one row per person**, with a column for each question.

**4a. Short names and clean answers.** We give each question a short name, then keep only *Yes* / *No* answers (dropping *Skip*, *Prefer not to answer*, and similar).

In [ ]:
survey_clean <- survey %>%
  mutate(
    question = case_when(
      str_detect(question, regex("prescribed|treatment", ignore_case = TRUE)) ~ "ptsd_treatment",
      str_detect(question, regex("doctor|provider", ignore_case = TRUE))      ~ "ptsd_doctor",
      TRUE ~ NA_character_
    ),
    answer = case_when(
      str_detect(answer, regex("\\byes\\b", ignore_case = TRUE)) ~ "Yes",
      str_detect(answer, regex("\\bno\\b",  ignore_case = TRUE)) ~ "No",
      TRUE ~ NA_character_
    )
  ) %>%
  filter(!is.na(question), !is.na(answer)) %>%
  distinct(person_id, question, .keep_all = TRUE)   # one answer per person per question

survey_clean %>% count(question, answer)

**4b. Long to wide.** `pivot_wider()` turns the `question` values into column names and the `answer` values into their contents.

In [ ]:
survey_wide <- survey_clean %>%
  pivot_wider(names_from = question, values_from = answer)

head(survey_wide)

## Step 5: Measurements

**5a. Group the measurement names.** There are many slightly different names for the same measurement (e.g. several kinds of systolic blood pressure). See for yourself:

In [ ]:
measurement %>% count(standard_concept_name, sort = TRUE)

We collapse them into four groups with `case_when()`. Order matters: the first matching rule wins.

In [ ]:
measurement_grouped <- measurement %>%
  mutate(measure_group = case_when(
    str_detect(standard_concept_name, regex("body mass index", ignore_case = TRUE))                     ~ "BMI",
    str_detect(standard_concept_name, regex("systolic", ignore_case = TRUE))                            ~ "Systolic_BP",
    str_detect(standard_concept_name, regex("a1c|glycated hemoglobin|glycohemoglobin", ignore_case = TRUE)) ~ "HbA1c",
    str_detect(standard_concept_name, regex("ldl", ignore_case = TRUE))                                 ~ "LDL_Cholesterol",
    TRUE ~ "Other"
  ))

measurement_grouped %>% count(measure_group)

**5b. Keep the most recent value** of each measurement for each person. `slice_max()` keeps the row with the latest date within each group.

In [ ]:
measurement_latest <- measurement_grouped %>%
  filter(measure_group != "Other") %>%
  group_by(person_id, measure_group) %>%
  slice_max(measurement_datetime, n = 1, with_ties = FALSE) %>%
  ungroup()

nrow(measurement_grouped)
nrow(measurement_latest)

**5c. Long to wide:** one column per measurement.

In [ ]:
measurements_wide <- measurement_latest %>%
  pivot_wider(id_cols = person_id, names_from = measure_group, values_from = value_as_number)

head(measurements_wide)

We save this table too: the survival analysis (3.4) uses it.

In [ ]:
save_data(measurements_wide, "measurements_wide.csv")

## Step 6: Join everything

Now we **join**: `left_join()` keeps every person in `person` (with their `diabetes` flag) and adds the survey and measurement columns that match on `person_id`. Then we:

- calculate **age** from date of birth
- make **age groups** (our population is 25 to 50)
- keep only people with **complete data** (`drop_na()`), so every analysis uses the same people

In [ ]:
final_analysis <- person %>%
  left_join(survey_wide,       by = "person_id") %>%
  left_join(measurements_wide, by = "person_id") %>%
  mutate(
    age = floor(time_length(interval(date_of_birth, today()), "years")),
    age_group = cut(age, breaks = c(25, 35, 45, 51), right = FALSE,
                    labels = c("25-34", "35-44", "45-50"))
  ) %>%
  select(person_id, diabetes, age, age_group, gender, race, ethnicity,
         ptsd_doctor, ptsd_treatment, BMI, Systolic_BP, LDL_Cholesterol, HbA1c) %>%
  drop_na()

glimpse(final_analysis)

> **Think about it:** `drop_na()` removes everyone missing *any* variable. How might the people we keep differ from the people we drop? (Complete-case analysis can introduce bias.)

## Step 7: Save

In [ ]:
save_data(final_analysis, "final_analysis.csv")

**Next:** 3.1 table_one.